# un-ai — KaniTTS ky (Kaggle T4 или Google Colab)

Kaggle отдаёт GPU только после **проверки телефона**. Если Accelerator = None — это не баг ноутбука.

**Kaggle:** аватар → Settings → Phone Verification → код из SMS → обновить страницу ноутбука → Session options → GPU T4 (или P100 / T4 x2) и Internet On.

**Если T4 так и нет — Colab (тот же файл):**
1. [colab.research.google.com](https://colab.research.google.com) → Upload notebook.
2. Вверху: **Среда выполнения** → **Сменить тип среды выполнения** → аппаратный ускоритель **T4 GPU** → **Сохранить**.
3. Справа сверху **Подключиться**. Затем **Среда выполнения** → **Выполнить все**. Скопируйте `KANI_BASE_URL` из последней ячейки.

Не запускайте отдельно `pip install kani-tts==1.0.1`: на Colab Python 3.13 pip часами перебирает `contourpy` / `ml-dtypes`.

Вкладку не закрывайте. Платный VPS не нужен.

In [ ]:
import os, torch
WORKDIR = "/kaggle/working" if os.path.isdir("/kaggle/working") else "/content"
print("WORKDIR", WORKDIR)
assert torch.cuda.is_available(), (
    "GPU нет. Kaggle: подтвердите телефон в Settings, затем Session options → GPU. "
    "Иначе Colab: Среда выполнения → Сменить тип среды выполнения → T4 GPU."
)
print("GPU:", torch.cuda.get_device_name(0), "VRAM", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1), "GB")

In [ ]:
# Colab 3.13: kani-tts без --no-deps тянет nemo-toolkit[tts] и pip часами бэктрекает contourpy.
# NeMo нужен только как кодек; extra [tts] (matplotlib/datasets/optuna) для голоса не нужен.
%pip install -q fastapi uvicorn soundfile "numpy>=1.26" "scipy>=1.11" librosa omegaconf einops hydra-core
%pip install -q --no-deps kani-tts==1.0.1
%pip install -q --no-deps "nemo-toolkit==2.4.0"
%pip install -q -U "transformers==4.57.1" "accelerate>=1.0.0"
import kani_tts
print("kani_tts OK", kani_tts.__file__)

In [ ]:
import os
from pathlib import Path

# Файлы ниже скопированы из apps/kani-server. Источник правды — репозиторий.
ROOT = Path("/kaggle/working") if Path("/kaggle/working").is_dir() else Path("/content")
ROOT.mkdir(parents=True, exist_ok=True)
os.chdir(ROOT)
print("cwd", Path.cwd())


In [ ]:
%%writefile config.py
"""Environment for the KaniTTS sidecar. No torch import here."""

from __future__ import annotations

import os
from dataclasses import dataclass
from pathlib import Path

DEFAULT_MODEL = "nineninesix/kani-tts-400m-ky"
DEFAULT_VOICE = "elina"
SAMPLE_RATE = 22050
ALLOWED_DEVICES = ("auto", "cpu", "cuda")


@dataclass(frozen=True)
class KaniSettings:
    model: str
    voice: str
    device: str
    device_map: str
    host: str
    port: int
    preload: bool
    api_key: str


def repo_root() -> Path:
    return Path(__file__).resolve().parents[2]


def _parse_env_line(line: str) -> tuple[str, str] | None:
    stripped = line.strip()
    if not stripped or stripped.startswith("#") or "=" not in stripped:
        return None
    key, value = stripped.split("=", 1)
    key = key.strip()
    if not key:
        return None
    value = value.strip()
    if len(value) >= 2 and value[0] == value[-1] and value[0] in {'"', "'"}:
        value = value[1:-1]
    return key, value


def load_env_file(path: Path, override: bool) -> None:
    if not path.is_file():
        return
    for line in path.read_text(encoding="utf-8").splitlines():
        parsed = _parse_env_line(line)
        if parsed is None:
            continue
        key, value = parsed
        if override or key not in os.environ:
            os.environ[key] = value


def load_repo_env() -> None:
    root = repo_root()
    load_env_file(root / ".env", override=False)
    load_env_file(root / ".env.local", override=True)


def _flag(value: str | None, default: bool) -> bool:
    if value is None or not value.strip():
        return default
    return value.strip().lower() not in {"0", "false", "no", "off"}


def normalize_device(raw: str | None) -> str:
    device = (raw or "auto").strip().lower()
    if device == "gpu":
        device = "cuda"
    if device not in ALLOWED_DEVICES:
        allowed = ", ".join(ALLOWED_DEVICES)
        raise ValueError(f"KANI_DEVICE must be {allowed}, got {raw!r}")
    return device


def load_settings(env: dict[str, str] | None = None, *, read_files: bool = True) -> KaniSettings:
    if env is None:
        if read_files:
            load_repo_env()
        env = dict(os.environ)
    device = normalize_device(env.get("KANI_DEVICE"))
    api_key = (env.get("KANI_API_KEY") or "").strip()
    if api_key == "not-needed":
        api_key = ""
    port_raw = (env.get("KANI_PORT") or "8090").strip()
    try:
        port = int(port_raw)
    except ValueError as exc:
        raise ValueError(f"KANI_PORT must be an integer, got {port_raw!r}") from exc
    if port <= 0 or port > 65535:
        raise ValueError(f"KANI_PORT out of range: {port}")
    model = (env.get("KANI_MODEL") or DEFAULT_MODEL).strip() or DEFAULT_MODEL
    voice = (env.get("KANI_VOICE") or DEFAULT_VOICE).strip() or DEFAULT_VOICE
    host = (env.get("KANI_HOST") or "0.0.0.0").strip() or "0.0.0.0"
    return KaniSettings(
        model=model,
        voice=voice,
        device=device,
        device_map="auto" if device == "auto" else device,
        host=host,
        port=port,
        preload=_flag(env.get("KANI_PRELOAD"), True),
        api_key=api_key,
    )


def prepare_process_device(device: str) -> None:
    """kani-tts 1.0.1 sends tensors to CUDA whenever it is visible.

    Hiding the devices is the only way KANI_DEVICE=cpu stays on CPU.
    Call this before the first torch import in the process.
    """
    if device == "cpu":
        os.environ["CUDA_VISIBLE_DEVICES"] = ""


def accepts_api_key(expected: str, authorization: str | None) -> bool:
    if not expected or expected == "not-needed":
        return True
    return authorization == f"Bearer {expected}"


def public_error(exc: BaseException) -> str:
    text = str(exc).strip() or exc.__class__.__name__
    lowered = text.lower()
    if "hf_" in lowered or "authorization" in lowered or "api_key" in lowered or "token=" in lowered:
        return "model load failed"
    return text[:500]


In [ ]:
%%writefile audio.py
"""WAV encoder for KaniTTS waveforms. The model returns 22.05 kHz mono floats."""

from __future__ import annotations

import io
import struct
import wave

from config import SAMPLE_RATE


def mono_floats(audio: object) -> list[float]:
    if hasattr(audio, "detach"):
        audio = audio.detach().cpu().numpy()  # type: ignore[union-attr]
    if hasattr(audio, "ndim") and getattr(audio, "ndim", None) == 2:
        shape = tuple(getattr(audio, "shape", ()))
        if 1 in shape:
            audio = audio.reshape(-1)  # type: ignore[union-attr]
    if hasattr(audio, "tolist"):
        data = audio.tolist()  # type: ignore[union-attr]
    else:
        data = list(audio)  # type: ignore[arg-type]
    if data and isinstance(data[0], (list, tuple)):
        raise ValueError("KaniTTS вернул неожиданную форму аудио")
    return [float(sample) for sample in data]


def to_wav_bytes(audio: object, sample_rate: int = SAMPLE_RATE) -> bytes:
    samples = mono_floats(audio)
    if not samples:
        raise ValueError("KaniTTS вернул пустое аудио")
    if sample_rate <= 0:
        raise ValueError("sample rate must be positive")
    pcm = bytearray()
    for sample in samples:
        clipped = max(-1.0, min(1.0, sample))
        pcm += struct.pack("<h", int(round(clipped * 32767)))
    buf = io.BytesIO()
    with wave.open(buf, "wb") as handle:
        handle.setnchannels(1)
        handle.setsampwidth(2)
        handle.setframerate(sample_rate)
        handle.writeframes(pcm)
    return buf.getvalue()


In [ ]:
%%writefile engine.py
"""Lazy KaniTTS loader. torch and kani-tts are imported on first use."""

from __future__ import annotations

import threading
from typing import Any

from config import SAMPLE_RATE, KaniSettings, public_error


def select_speaker(voice: str, status: str | None, speakers: list[str] | None) -> str | None:
    """Pass speaker_id only for the multispeaker Kyrgyz checkpoint.

    A single-speaker model still accepts the argument and then prefixes the
    text with the name, which changes the utterance. Skip it in that case.
    """
    if status != "multispeaker":
        return None
    names = [str(item).strip() for item in (speakers or []) if str(item).strip()]
    requested = voice.strip()
    if requested and requested in names:
        return requested
    return names[0] if names else None


class EnginePool:
    def __init__(self, settings: KaniSettings) -> None:
        self.settings = settings
        self._engine: Any = None
        self._load_lock = threading.Lock()
        self._infer_lock = threading.Lock()
        self.state = "idle"
        self.detail = ""
        self.device_actual = settings.device
        self.sample_rate = SAMPLE_RATE

    def snapshot(self) -> dict[str, str]:
        return {
            "status": self.public_status(),
            "service": "un-ai-kani",
            "model": self.settings.model,
            "device": self.device_actual,
            "voice": self.settings.voice,
            "ready": "yes" if self.state == "ok" else "no",
            "detail": self.detail,
        }

    def public_status(self) -> str:
        if self.state == "ok":
            return "ok"
        if self.state == "loading":
            return "loading"
        if self.state == "missing":
            return "missing_kani_tts"
        if self.state == "error":
            return "error"
        return "idle"

    def preload_async(self) -> None:
        if self.state == "ok":
            return
        threading.Thread(target=self._load_safe, name="kani-preload", daemon=True).start()

    def _load_safe(self) -> None:
        try:
            self.load()
        except Exception as exc:
            if self.state not in {"missing", "error"}:
                self.state = "error"
                self.detail = public_error(exc)

    def load(self) -> Any:
        with self._load_lock:
            if self._engine is not None:
                return self._engine
            self.state = "loading"
            self.detail = ""
            try:
                engine = self._build()
            except ImportError as exc:
                self.state = "missing"
                self.detail = public_error(exc)
                raise
            except Exception as exc:
                self.state = "error"
                self.detail = public_error(exc)
                raise
            self._engine = engine
            rate = getattr(engine, "sample_rate", SAMPLE_RATE)
            self.sample_rate = int(rate or SAMPLE_RATE)
            self.state = "ok"
            return engine

    def _build(self) -> Any:
        import torch
        from kani_tts import KaniTTS

        if self.settings.device == "cuda" and not torch.cuda.is_available():
            raise RuntimeError("KANI_DEVICE=cuda, но CUDA недоступна")
        engine = KaniTTS(
            self.settings.model,
            device_map=self.settings.device_map,
            show_info=True,
            suppress_logs=True,
        )
        if self.settings.device == "cpu" or not torch.cuda.is_available():
            self.device_actual = "cpu"
        else:
            self.device_actual = "cuda"
        return engine

    def speak(self, text: str, voice: str) -> tuple[Any, str]:
        engine = self.load()
        speaker = select_speaker(
            voice,
            getattr(engine, "status", None),
            list(getattr(engine, "speaker_list", []) or []),
        )
        with self._infer_lock:
            if speaker:
                return engine(text, speaker_id=speaker)
            return engine(text)


In [ ]:
%%writefile server.py
"""OpenAI-compatible KaniTTS sidecar for Kyrgyz replies.

The Node agent POSTs /v1/audio/speech. Russian stays on Cartesia and never
calls this process. If the model is not ready, /health is not "ok" and the
agent keeps the existing on-screen Kyrgyz reply.
"""

from __future__ import annotations

import logging
from contextlib import asynccontextmanager
from typing import Optional

from fastapi import FastAPI, HTTPException, Request
from fastapi.responses import Response
from pydantic import BaseModel, Field

from audio import to_wav_bytes
from config import accepts_api_key, load_settings, prepare_process_device, public_error
from engine import EnginePool

logger = logging.getLogger("un-ai.kani")

settings = load_settings()
prepare_process_device(settings.device)
pool = EnginePool(settings)


@asynccontextmanager
async def lifespan(_app: FastAPI):
    if settings.preload:
        logger.info("loading %s on %s", settings.model, settings.device)
        pool.preload_async()
    yield


app = FastAPI(title="un-ai kani-tts", lifespan=lifespan)


class SpeechRequest(BaseModel):
    input: str
    model: Optional[str] = None
    voice: Optional[str] = Field(default=None)
    response_format: Optional[str] = "wav"


@app.get("/health")
def health() -> dict[str, str]:
    return pool.snapshot()


@app.post("/v1/audio/speech")
def speech(body: SpeechRequest, request: Request) -> Response:
    if not accepts_api_key(settings.api_key, request.headers.get("authorization")):
        raise HTTPException(status_code=401, detail="invalid api key")
    text = body.input.strip()
    if not text:
        raise HTTPException(status_code=400, detail="empty input")
    if body.model and body.model not in {settings.model, "tts-1", "tts-1-hd"}:
        logger.warning("request model %s ignored; loaded %s", body.model, settings.model)
    voice = (body.voice or settings.voice).strip() or settings.voice
    try:
        audio, _generated = pool.speak(text, voice)
        wav = to_wav_bytes(audio, pool.sample_rate)
    except Exception as exc:
        raise HTTPException(status_code=500, detail=f"KaniTTS: {public_error(exc)}") from exc
    return Response(
        content=wav,
        media_type="audio/wav",
        headers={
            "X-Sample-Rate": str(pool.sample_rate),
            "X-Voice": voice,
            "X-Model": settings.model,
            "X-Device": pool.device_actual,
        },
    )


if __name__ == "__main__":
    import uvicorn

    uvicorn.run(
        "server:app",
        host=settings.host,
        port=settings.port,
        log_level="info",
    )


In [ ]:
import os, re, subprocess, sys, time, urllib.request

import kani_tts  # noqa: F401 — без пакета туннель бесполезен

PORT = 8765
WORKDIR = "/kaggle/working" if os.path.isdir("/kaggle/working") else "/content"
os.environ.setdefault("KANI_MODEL", "nineninesix/kani-tts-400m-ky")
os.environ.setdefault("KANI_VOICE", "elina")
os.environ["KANI_DEVICE"] = "cuda"
os.environ.setdefault("KANI_PRELOAD", "1")
os.chdir(WORKDIR)

bin_path = f"{WORKDIR}/cloudflared"
if not os.path.exists(bin_path):
    print("скачиваю cloudflared…")
    urllib.request.urlretrieve(
        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
        bin_path,
    )
    os.chmod(bin_path, 0o755)

uvicorn_log = open(f"{WORKDIR}/uvicorn.log", "w")
tunnel_log_path = f"{WORKDIR}/tunnel.log"
tunnel_log = open(tunnel_log_path, "w")

uvicorn = subprocess.Popen(
    [sys.executable, "-m", "uvicorn", "server:app", "--host", "127.0.0.1", "--port", str(PORT)],
    cwd=WORKDIR,
    stdout=uvicorn_log,
    stderr=subprocess.STDOUT,
)
# HTTP/2: на Kaggle QUIC/UDP часто режут, из‑за этого туннель «молчит».
tunnel = subprocess.Popen(
    [bin_path, "tunnel", "--url", f"http://127.0.0.1:{PORT}", "--protocol", "http2", "--no-autoupdate"],
    stdout=tunnel_log,
    stderr=subprocess.STDOUT,
)

url = None
deadline = time.time() + 90
pattern = re.compile(r"https://[a-z0-9-]+\.trycloudflare\.com")
print("жду URL туннеля (до 90 сек)…")
while time.time() < deadline and url is None:
    time.sleep(2)
    text = open(tunnel_log_path, encoding="utf-8", errors="replace").read()
    found = pattern.search(text)
    if found:
        url = found.group(0)
    if uvicorn.poll() is not None:
        print("uvicorn упал:\n", open(f"{WORKDIR}/uvicorn.log", encoding="utf-8", errors="replace").read()[-4000:])
        break
    if tunnel.poll() is not None and url is None:
        print("cloudflared вышел:\n", text[-4000:])
        break

if not url:
    print("--- cloudflared.log ---")
    print(open(tunnel_log_path, encoding="utf-8", errors="replace").read()[-4000:])
    raise RuntimeError(
        "Туннель не поднялся. Нужны Settings → Internet: On и GPU T4. "
        "Пришлите хвост cloudflared.log из вывода выше."
    )

print("\n--- вставьте в .env на своём ПК ---")
print(f"KANI_BASE_URL={url}/v1")
print("KANI_MODEL=nineninesix/kani-tts-400m-ky")
print("KANI_VOICE=elina")
print("-----------------------------------")
print("жду /health status=ok (веса, 1–3 мин). Звоните только после строки «можно звонить».")

import json as _json
health = None
health_deadline = time.time() + 600
while time.time() < health_deadline:
    if uvicorn.poll() is not None:
        print(open(f"{WORKDIR}/uvicorn.log", encoding="utf-8", errors="replace").read()[-4000:])
        raise RuntimeError("uvicorn упал до загрузки модели")
    try:
        with urllib.request.urlopen(f"http://127.0.0.1:{PORT}/health", timeout=5) as resp:
            health = _json.loads(resp.read().decode())
    except Exception as exc:
        health = {"status": "down", "detail": str(exc)}
    status = str(health.get("status"))
    print("health", status, health.get("detail") or "")
    if status == "ok":
        print("можно звонить. Ноутбук не закрывайте.")
        break
    if status in {"error", "missing_kani_tts"}:
        raise RuntimeError(f"KaniTTS не поднялся: {health}")
    time.sleep(5)
else:
    raise RuntimeError(f"модель не стала ok за 10 мин: {health}")

try:
    while True:
        time.sleep(120)
        if uvicorn.poll() is not None or tunnel.poll() is not None:
            print("процесс упал, перезапустите ячейку")
            print(open(tunnel_log_path, encoding="utf-8", errors="replace").read()[-2000:])
            break
except KeyboardInterrupt:
    uvicorn.terminate()
    tunnel.terminate()
